# Notebook 03 — Real-Data Verification

**Experiment 2** from the proposal: apply the trained CNN to ~100 000 real
candlestick windows from 50 S&P 500 stocks (2015–2022) and measure whether
the detected patterns actually predict the subsequent 5-day price movement.

**Key question:** *Do chart patterns actually predict price direction?*

Metrics:
- Directional accuracy (baseline = 50 %)
- One-sample t-test vs 50 % (is the difference statistically significant?)
- Per-class breakdown


In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import *
from src.models.cnn           import PatternCNN
from src.verification.stock_data import run_verification, SP500_TICKERS
from src.verification.backtest  import directional_accuracy, per_class_stats

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load trained model

In [ ]:
CKPT_PATH = os.path.join(ROOT, 'checkpoints', 'cnn_best.pt')

model = PatternCNN(num_classes=7)
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE).eval()

print(f'Loaded checkpoint (val_acc = {checkpoint["val_acc"]:.4f})')

## 2. Run sliding-window detection over all stocks

> **Runtime estimate:** ~1–3 hours for 50 tickers on CPU.  
> Reduce `TICKERS` list or increase `STEP_SIZE_SLIDE` for faster runs.

In [ ]:
# ── Use a subset for quick testing; set to SP500_TICKERS for full run ─────────
TICKERS_TO_USE = SP500_TICKERS   # or e.g. SP500_TICKERS[:10]

detections_df = run_verification(
    model,
    DEVICE,
    tickers              = TICKERS_TO_USE,
    start                = VERIFY_START,
    end                  = VERIFY_END,
    window_size          = WINDOW_SIZE,
    step_size            = STEP_SIZE_SLIDE,
    confidence_threshold = CONFIDENCE_THRESHOLD,
    forward_days         = FORWARD_DAYS,
    verbose              = True,
)

# Save detections to CSV for inspection / re-use
csv_path = os.path.join(ROOT, 'results', 'logs', 'detections.csv')
detections_df.to_csv(csv_path, index=False)
print(f'Saved {len(detections_df)} detections to {csv_path}')

detections_df.head(10)

## 3. Directional accuracy & statistical significance

In [ ]:
acc, t_stat, p_value, n_trades = directional_accuracy(
    detections_df, forward_col=f'forward_return_{FORWARD_DAYS}d'
)

print('─' * 50)
print(f'Detections (pattern classes only): {n_trades:,}')
print(f'Directional accuracy:  {acc:.2%}  (baseline 50.00 %)')
print(f't-statistic vs 50 %:   {t_stat:+.3f}')
print(f'p-value:               {p_value:.4f}')
print(f'Statistically significant (p<0.05): {"YES" if p_value < 0.05 else "NO"}')
print('─' * 50)

## 4. Per-class breakdown

In [ ]:
per_class = per_class_stats(detections_df, forward_col=f'forward_return_{FORWARD_DAYS}d')
print(per_class[['n', 'directional_accuracy', 'mean_return', 'p_value', 'direction']].to_string())

# Save for report
per_class.to_csv(os.path.join(ROOT, 'results', 'logs', 'per_class_stats.csv'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar: directional accuracy per class
valid_cls = per_class[per_class['n'] > 0]
colors = ['#ef5350' if d == 'bearish' else '#26a69a'
          for d in valid_cls['direction']]

axes[0].bar(range(len(valid_cls)),
            valid_cls['directional_accuracy'].values,
            color=colors, edgecolor='black')
axes[0].axhline(0.5, color='black', linestyle='--', label='Baseline (50 %)')
axes[0].set_xticks(range(len(valid_cls)))
axes[0].set_xticklabels(valid_cls.index, rotation=30, ha='right')
axes[0].set_ylabel('Directional accuracy')
axes[0].set_title('Directional accuracy by pattern class')
axes[0].legend()

# Bar: mean 5-day forward return
axes[1].bar(range(len(valid_cls)),
            valid_cls['mean_return'].values * 100,
            color=colors, edgecolor='black')
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xticks(range(len(valid_cls)))
axes[1].set_xticklabels(valid_cls.index, rotation=30, ha='right')
axes[1].set_ylabel('Mean 5-day forward return (%)')
axes[1].set_title('Mean forward return by pattern class')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'verification_results.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 5. Detection frequency by class

In [ ]:
freq = detections_df['class_name'].value_counts()
freq.plot(kind='bar', figsize=(10, 4), color='steelblue', edgecolor='black')
plt.title('Number of detections per pattern class (real data)')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'detection_frequency.png'),
            dpi=150, bbox_inches='tight')
plt.show()